# Canadian Cheese vs Provincial Climate: ETL-Driven Assessment

## Executive Summary
This notebook evaluates whether provincial weather patterns are associated with the types and characteristics of cheeses produced in Canada. The workflow is intentionally built like a production analytics pipeline: robust data extraction, strict cleaning and standardization, transparent transformation logic, and validated joins before any interpretation.

The analysis integrates `cheese_data.csv` with both weather sources (`canada_weather.csv` and `Canada_Temperature_Data.csv.zip`) to construct a reliable province-level **Average Annual Temperature** feature. Two interactive visualizations and an executive discussion then answer the core business question: does provincial climate meaningfully shape cheese production, or do regional culture and market history dominate?

In [1]:
import re
import unicodedata
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

## Pipeline Architecture (ETL)

This notebook follows a modular ETL design:

1. **Extract**: Load all three raw datasets from CSV/ZIP sources.
2. **Transform**:
   - Standardize province values with regex-based cleanup and name harmonization (including English/French variants).
   - Clean and impute critical analytical features (`MoisturePercent`, fat proxy, and temperature fields).
   - Build one province-level climate metric by combining two independent weather datasets.
3. **Load (Analytical Layer)**: Merge cleaned cheese records with cleaned province temperature aggregates and validate merge integrity.

This structure is production-minded: auditable, reusable, and robust against schema inconsistencies.

In [2]:
# -----------------------------
# Constants and mappings
# -----------------------------
PROVINCE_CODE_TO_NAME: Dict[str, str] = {
    "AB": "Alberta",
    "BC": "British Columbia",
    "MB": "Manitoba",
    "NB": "New Brunswick",
    "NL": "Newfoundland and Labrador",
    "NS": "Nova Scotia",
    "NT": "Northwest Territories",
    "NU": "Nunavut",
    "ON": "Ontario",
    "PE": "Prince Edward Island",
    "QC": "Quebec",
    "SK": "Saskatchewan",
    "YT": "Yukon",
}

PROVINCE_NAME_ALIASES: Dict[str, str] = {
    "alberta": "Alberta",
    "british columbia": "British Columbia",
    "colombie britannique": "British Columbia",
    "manitoba": "Manitoba",
    "new brunswick": "New Brunswick",
    "nouveau brunswick": "New Brunswick",
    "newfoundland and labrador": "Newfoundland and Labrador",
    "terre neuve and labrador": "Newfoundland and Labrador",
    "terre neuve et labrador": "Newfoundland and Labrador",
    "nova scotia": "Nova Scotia",
    "nouvelle ecosse": "Nova Scotia",
    "ontario": "Ontario",
    "prince edward island": "Prince Edward Island",
    "ile du prince edouard": "Prince Edward Island",
    "quebec": "Quebec",
    "saskatchewan": "Saskatchewan",
    "northwest territories": "Northwest Territories",
    "territoires du nord ouest": "Northwest Territories",
    "nunavut": "Nunavut",
    "yukon": "Yukon",
}

PROVINCE_CODE_ALIASES: Dict[str, str] = {
    "PQ": "QC",
}


def normalize_text(value: str) -> str:
    if pd.isna(value):
        return np.nan
    txt = str(value).strip()
    txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode("utf-8")
    txt = txt.lower()
    txt = re.sub(r"[^a-z0-9\s\-&]", " ", txt)
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt


def normalize_province_name(name: str) -> str:
    key = normalize_text(name)
    if pd.isna(key):
        return np.nan
    return PROVINCE_NAME_ALIASES.get(key, str(name).strip().title())


def parse_celsius_from_text(value: str) -> float:
    if pd.isna(value):
        return np.nan
    cleaned = str(value).replace("−", "-")
    match = re.search(r"-?\d+(?:\.\d+)?", cleaned)
    return float(match.group()) if match else np.nan


def classify_milk_type(milk: str) -> str:
    if pd.isna(milk):
        return "Unknown"
    m = str(milk).lower()
    has_cow = "cow" in m
    has_goat = "goat" in m
    has_ewe = ("ewe" in m) or ("sheep" in m)
    has_buffalo = "buffalo" in m

    components = sum([has_cow, has_goat, has_ewe, has_buffalo])
    if components > 1:
        return "Mixed"
    if has_ewe:
        return "Sheep"
    if has_goat:
        return "Goat"
    if has_cow:
        return "Cow"
    if has_buffalo:
        return "Buffalo"
    return "Other"


def load_data(
    cheese_path: str = "cheese_data.csv",
    weather_path: str = "canada_weather.csv",
    temp_zip_path: str = "Canada_Temperature_Data.csv.zip",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load the three input datasets."""
    cheese_df = pd.read_csv(cheese_path)
    weather_df = pd.read_csv(weather_path)
    temp_df = pd.read_csv(temp_zip_path, compression="zip")
    return cheese_df, weather_df, temp_df


def clean_cheese_data(cheese_df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize province values and clean core cheese features.
    Includes null handling for Moisture and engineered numeric fat proxy.
    """
    df = cheese_df.copy()

    # Standardize province code and remove non-letter noise.
    df["ProvinceCode"] = (
        df["ManufacturerProvCode"]
        .astype(str)
        .str.upper()
        .str.strip()
        .replace(PROVINCE_CODE_ALIASES)
        .str.replace(r"[^A-Z]", "", regex=True)
    )

    # Build canonical province names.
    df["Province"] = df["ProvinceCode"].map(PROVINCE_CODE_TO_NAME)
    if "Province" in df.columns:
        df["Province"] = df["Province"].fillna(df["Province"].apply(normalize_province_name))

    # Clean moisture and impute nulls province-first, then global median fallback.
    df["MoisturePercent"] = pd.to_numeric(df["MoisturePercent"], errors="coerce")
    df["MoisturePercent"] = df.groupby("ProvinceCode")["MoisturePercent"].transform(
        lambda s: s.fillna(s.median())
    )
    df["MoisturePercent"] = df["MoisturePercent"].fillna(df["MoisturePercent"].median())

    # FatLevel is categorical in source; map to transparent analytical proxy for bubble size/color.
    df["FatLevelClean"] = df["FatLevel"].astype(str).str.lower().str.strip()
    fat_proxy_map = {"lower fat": 25.0, "higher fat": 45.0}
    df["EstimatedFatPercent"] = df["FatLevelClean"].map(fat_proxy_map).fillna(35.0)

    # Standardized milk type groups for downstream stacked bar readability.
    df["MilkTypeGroup"] = df["MilkTypeEn"].apply(classify_milk_type)

    # Keep analytically valid records.
    df = df.dropna(subset=["CheeseName", "ProvinceCode", "Province"]).reset_index(drop=True)
    return df


def clean_weather_data(weather_df: pd.DataFrame, temp_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build one reliable Average Annual Temperature per province by combining:
    - canada_weather.csv (community annual high/low summaries)
    - Canada_Temperature_Data.csv (historical station monthly mean temperatures)
    """
    # Source A: community-level weather file
    w = weather_df.copy()
    annual_high_col = [c for c in w.columns if "Annual(Avg. high" in c][0]
    annual_low_col = [c for c in w.columns if "Annual(Avg. low" in c][0]

    w["AnnualHighC"] = w[annual_high_col].apply(parse_celsius_from_text)
    w["AnnualLowC"] = w[annual_low_col].apply(parse_celsius_from_text)
    w["WeatherAnnualMeanC"] = (w["AnnualHighC"] + w["AnnualLowC"]) / 2
    w["ProvinceCode"] = (
        w["Community"].astype(str).str.extract(r",\s*([A-Z]{2})\s*$")[0].str.upper().str.strip()
    )

    weather_agg = (
        w.dropna(subset=["ProvinceCode", "WeatherAnnualMeanC"])
        .groupby("ProvinceCode", as_index=False)["WeatherAnnualMeanC"]
        .mean()
    )

    # Source B: historical station temperatures
    t = temp_df.copy()
    t["ProvinceCode"] = t["Prov"].astype(str).str.upper().str.strip()
    t["Tm"] = pd.to_numeric(t["Tm"], errors="coerce")

    # Filter obvious outliers and use modern climatology period for better comparability.
    t = t[t["Tm"].between(-50, 40, inclusive="both")]
    if "Year" in t.columns:
        t = t[t["Year"] >= 1991]

    temp_agg = (
        t.dropna(subset=["ProvinceCode", "Tm"])
        .groupby("ProvinceCode", as_index=False)["Tm"]
        .mean()
        .rename(columns={"Tm": "StationAnnualMeanC"})
    )

    # Combined provincial estimate: average of available source means.
    merged = temp_agg.merge(weather_agg, on="ProvinceCode", how="outer")
    merged["AverageAnnualTemperature"] = merged[["StationAnnualMeanC", "WeatherAnnualMeanC"]].mean(axis=1)
    merged["Province"] = merged["ProvinceCode"].map(PROVINCE_CODE_TO_NAME)

    merged = merged.dropna(subset=["ProvinceCode", "AverageAnnualTemperature", "Province"]).copy()
    merged = merged.sort_values("AverageAnnualTemperature").reset_index(drop=True)
    return merged


# Execute ETL
cheese_raw, weather_raw, temp_raw = load_data()
cheese_clean = clean_cheese_data(cheese_raw)
weather_clean = clean_weather_data(weather_raw, temp_raw)

print("=== Clean Cheese Data: info() ===")
cheese_clean.info()
print("\n=== Clean Cheese Data: head() ===")
display(cheese_clean.head())

print("\n=== Clean Weather Aggregate: info() ===")
weather_clean.info()
print("\n=== Clean Weather Aggregate: head() ===")
display(weather_clean.head())

=== Clean Cheese Data: info() ===
<class 'pandas.DataFrame'>
RangeIndex: 1042 entries, 0 to 1041
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   CheeseId              1042 non-null   int64  
 1   ManufacturerProvCode  1042 non-null   str    
 2   ManufacturingTypeEn   1042 non-null   str    
 3   MoisturePercent       1042 non-null   float64
 4   FlavourEn             801 non-null    str    
 5   CharacteristicsEn     643 non-null    str    
 6   Organic               1042 non-null   int64  
 7   CategoryTypeEn        1019 non-null   str    
 8   MilkTypeEn            1041 non-null   str    
 9   MilkTreatmentTypeEn   977 non-null    str    
 10  RindTypeEn            721 non-null    str    
 11  CheeseName            1042 non-null   str    
 12  FatLevel              1042 non-null   str    
 13  ProvinceCode          1042 non-null   str    
 14  Province              1042 non-null   str    
 15

,CheeseId,ManufacturerProvCode,ManufacturingTypeEn,MoisturePercent,FlavourEn,CharacteristicsEn,Organic,CategoryTypeEn,MilkTypeEn,MilkTreatmentTypeEn,RindTypeEn,CheeseName,FatLevel,ProvinceCode,Province,FatLevelClean,EstimatedFatPercent,MilkTypeGroup
0,228,NB,Farmstead,47.0,"Sharp, lactic",Uncooked,0,Firm Cheese,Ewe,Raw Milk,Washed Rind,Sieur de Duplessis (Le),lower fat,NB,New Brunswick,lower fat,25.0,Sheep
1,242,NB,Farmstead,47.9,"Sharp, lactic, lightly caramelized",Uncooked,0,Semi-soft Cheese,Cow,Raw Milk,Washed Rind,Tomme Le Champ Doré,lower fat,NB,New Brunswick,lower fat,25.0,Cow
2,301,ON,Industrial,54.0,"Mild, tangy, and fruity","Pressed and cooked cheese, pasta filata, inter...",0,Firm Cheese,Cow,Pasteurized,NaN,Provolone Sette Fette (Tre-Stelle),lower fat,ON,Ontario,lower fat,25.0,Cow
3,303,NB,Farmstead,47.0,Sharp with fruity notes and a hint of wild honey,NaN,0,Veined Cheeses,Cow,Raw Milk,NaN,Geai Bleu (Le),lower fat,NB,New Brunswick,lower fat,25.0,Cow
4,319,NB,Farmstead,49.4,Softer taste,NaN,1,Semi-soft Cheese,Cow,Raw Milk,Washed Rind,Gamin (Le),lower fat,NB,New Brunswick,lower fat,25.0,Cow



=== Clean Weather Aggregate: info() ===
<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ProvinceCode              13 non-null     str    
 1   StationAnnualMeanC        13 non-null     float64
 2   WeatherAnnualMeanC        13 non-null     float64
 3   AverageAnnualTemperature  13 non-null     float64
 4   Province                  13 non-null     str    
dtypes: float64(3), str(2)
memory usage: 652.0 bytes

=== Clean Weather Aggregate: head() ===


,ProvinceCode,StationAnnualMeanC,WeatherAnnualMeanC,AverageAnnualTemperature,Province
0,NU,-11.812795,-11.625000,-11.718898,Nunavut
1,NT,-5.065504,-5.883333,-5.474419,Northwest Territories
2,YT,-2.614988,-2.183333,-2.399161,Yukon
3,MB,1.961404,-2.116667,-0.077631,Manitoba
4,SK,2.547995,1.933333,2.240664,Saskatchewan


In [3]:
def merge_datasets(cheese_df: pd.DataFrame, weather_df: pd.DataFrame) -> pd.DataFrame:
    """Merge cleaned datasets and print integrity checks before/after join."""
    print("Cheese DF shape before merge:", cheese_df.shape)
    print("Weather DF shape before merge:", weather_df.shape)

    merged_df = cheese_df.merge(
        weather_df[["ProvinceCode", "AverageAnnualTemperature", "Province"]],
        on="ProvinceCode",
        how="left",
        validate="many_to_one",
        suffixes=("", "_weather")
    )

    # Keep the cheese-side canonical province where available.
    merged_df["Province"] = merged_df["Province"].fillna(merged_df["Province_weather"])
    merged_df = merged_df.drop(columns=["Province_weather"])

    print("Merged DF shape:", merged_df.shape)
    print("Rows with missing AverageAnnualTemperature:", merged_df["AverageAnnualTemperature"].isna().sum())

    # Data integrity assertion: left join must preserve all cheese rows.
    assert merged_df.shape[0] == cheese_df.shape[0], "Row count changed after merge."

    return merged_df


analysis_df = merge_datasets(cheese_clean, weather_clean)
display(analysis_df.head())

Cheese DF shape before merge: (1042, 18)
Weather DF shape before merge: (13, 5)
Merged DF shape: (1042, 19)
Rows with missing AverageAnnualTemperature: 0


,CheeseId,ManufacturerProvCode,ManufacturingTypeEn,MoisturePercent,FlavourEn,CharacteristicsEn,Organic,CategoryTypeEn,MilkTypeEn,MilkTreatmentTypeEn,RindTypeEn,CheeseName,FatLevel,ProvinceCode,Province,FatLevelClean,EstimatedFatPercent,MilkTypeGroup,AverageAnnualTemperature
0,228,NB,Farmstead,47.0,"Sharp, lactic",Uncooked,0,Firm Cheese,Ewe,Raw Milk,Washed Rind,Sieur de Duplessis (Le),lower fat,NB,New Brunswick,lower fat,25.0,Sheep,4.882356
1,242,NB,Farmstead,47.9,"Sharp, lactic, lightly caramelized",Uncooked,0,Semi-soft Cheese,Cow,Raw Milk,Washed Rind,Tomme Le Champ Doré,lower fat,NB,New Brunswick,lower fat,25.0,Cow,4.882356
2,301,ON,Industrial,54.0,"Mild, tangy, and fruity","Pressed and cooked cheese, pasta filata, inter...",0,Firm Cheese,Cow,Pasteurized,NaN,Provolone Sette Fette (Tre-Stelle),lower fat,ON,Ontario,lower fat,25.0,Cow,6.445578
3,303,NB,Farmstead,47.0,Sharp with fruity notes and a hint of wild honey,NaN,0,Veined Cheeses,Cow,Raw Milk,NaN,Geai Bleu (Le),lower fat,NB,New Brunswick,lower fat,25.0,Cow,4.882356
4,319,NB,Farmstead,49.4,Softer taste,NaN,1,Semi-soft Cheese,Cow,Raw Milk,Washed Rind,Gamin (Le),lower fat,NB,New Brunswick,lower fat,25.0,Cow,4.882356


## Visualization 1 Introduction

The first chart examines how cheese moisture and fat intensity vary as provincial average annual temperature changes.

In [4]:
viz1_df = analysis_df.dropna(subset=["AverageAnnualTemperature", "MoisturePercent"]).copy()

fig1 = px.scatter(
    viz1_df,
    x="AverageAnnualTemperature",
    y="MoisturePercent",
    size="EstimatedFatPercent",
    color="EstimatedFatPercent",
    color_continuous_scale="Viridis",
    hover_data={
        "CheeseName": True,
        "MilkTypeEn": True,
        "Province": True,
        "FatLevelClean": True,
        "AverageAnnualTemperature": ":.2f",
        "MoisturePercent": ":.2f",
        "EstimatedFatPercent": ":.1f",
    },
    title="Moisture and Fat Profile vs Provincial Average Annual Temperature",
    labels={
        "AverageAnnualTemperature": "Average Provincial Temperature (C)",
        "MoisturePercent": "Moisture Percentage",
        "EstimatedFatPercent": "Estimated Fat Percentage",
    },
    template="plotly_white",
    opacity=0.78,
)

fig1.update_traces(marker=dict(line=dict(width=0.5, color="white")))
fig1.update_layout(coloraxis_colorbar_title="Est. Fat %")
fig1.show()

## Visualization 2 Introduction

The second chart shows the milk-type composition of each province, ordered from coldest to warmest average climate.

In [5]:
province_order = (
    weather_clean[weather_clean["ProvinceCode"].isin(analysis_df["ProvinceCode"].unique())]
    .sort_values("AverageAnnualTemperature")["Province"]
    .tolist()
)

viz2_df = (
    analysis_df.groupby(["Province", "MilkTypeGroup"], as_index=False)
    .size()
    .rename(columns={"size": "CheeseCount"})
)

viz2_df["Province"] = pd.Categorical(viz2_df["Province"], categories=province_order, ordered=True)
viz2_df = viz2_df.sort_values("Province")

fig2 = px.bar(
    viz2_df,
    x="Province",
    y="CheeseCount",
    color="MilkTypeGroup",
    barmode="stack",
    title="Milk Type Distribution by Climate: Canadian Cheese Production",
    labels={
        "Province": "Province (Ordered from Lowest to Highest Average Temperature)",
        "CheeseCount": "Count of Cheeses Produced",
        "MilkTypeGroup": "Milk Type",
    },
    template="plotly_white",
)

fig2.update_layout(xaxis_tickangle=-35)
fig2.show()

## Executive Inferences and Discussion

Across provinces, temperature shows a visible but generally modest association with cheese characteristics rather than a strict deterministic pattern. In this sample, cooler provinces include many cheeses with mid-to-higher moisture values, while warmer provinces show a broad spread of moisture and fat profiles. This suggests climate may influence production constraints and affinage conditions indirectly, but it does not act as a single dominant predictor of final cheese style.

A stronger signal comes from provincial production ecosystems, especially concentration effects. Quebec contributes the majority of records and exhibits substantial internal diversity across milk sources and style profiles, which weakens simple climate-only interpretations. Ontario and British Columbia also show multi-segment portfolios, indicating that supply chain maturity, producer specialization, and market demand likely shape product mix at least as much as annual temperature.

From a senior analytical perspective, climate should be treated as an enabling context variable, not a standalone causal driver. Biological and environmental factors can influence feed patterns, milk quality, and storage/aging dynamics, but historical cheese-making traditions, regulatory environment, and regional culinary identity (notably in Quebec) appear to be the dominant structural forces. The business inference is to model cheese outcomes using climate together with cultural and industrial covariates, rather than relying on weather alone.